
## One-off: generate supplementary guts database

Run the following in a bash terminal:

```bash
guts manifest -c singularity -i fs -i paths -o miniconda.json anaconda/miniconda
```

This produces the `miniconda` manifest json with all the paths. This json will be bundled with shelley-bio so `extract_aliases` can subtract conda binaries at runtime without any extra network calls.


## Trace `extract_aliases` step by step

Walk through each stage of the alias-generation pipeline for a CVMFS tool.

In [1]:
from shelley_bio.builder.cvmfs_builder import CVMFSModuleBuilder
b = CVMFSModuleBuilder()

In [2]:
# Step 1 — pick a tool version and build the CVMFS container path
import os
from pathlib import Path

tool_name = "star-fusion"
versions = b.list_versions(tool_name)
print("Available versions:", versions[:5])

v = '1.0.0--pl5.22.0_0' # older version
cvmfs_path = str(b.cvmfs_singularity_path / f"{tool_name}:{v}")
print(f"\nCVMFS path: {cvmfs_path}")
print(f"Exists on disk: {Path(cvmfs_path).exists()}")

KeyboardInterrupt: 

In [ ]:
# Step 2 — sparse-clone base OS manifests from shpc-guts
import subprocess, tempfile, shutil

from shelley_bio.builder.guts_integration import (
    _sparse_clone_base_manifests,
    SHPC_GUTS_DB_URL,
    BASE_IMAGE_NAMESPACES,
)

tmpdir = _sparse_clone_base_manifests(SHPC_GUTS_DB_URL, BASE_IMAGE_NAMESPACES)
print(f"Cloned to: {tmpdir}")

cloned = list(Path(tmpdir).rglob("*.json"))
print(f"\nJSON manifest files ({len(cloned)} total):")
for p in sorted(cloned):
    print(" ", p.relative_to(tmpdir))

Cloned to: /tmp/shelley-guts-j_dabz64

JSON manifest files (32 total):
  docker.io/library/alpine/3.10.json
  docker.io/library/alpine/3.11.json
  docker.io/library/alpine/3.12.json
  docker.io/library/alpine/3.13.json
  docker.io/library/alpine/3.14.json
  docker.io/library/alpine/3.15.json
  docker.io/library/alpine/3.16.json
  docker.io/library/alpine/3.17.json
  docker.io/library/alpine/3.18.json
  docker.io/library/alpine/3.19.json
  docker.io/library/alpine/3.20.json
  docker.io/library/alpine/3.7.json
  docker.io/library/alpine/3.8.json
  docker.io/library/alpine/3.9.json
  docker.io/library/busybox/1.23.json
  docker.io/library/busybox/1.24.json
  docker.io/library/busybox/1.25.json
  docker.io/library/busybox/1.26.json
  docker.io/library/busybox/1.27.json
  docker.io/library/busybox/1.28.json
  docker.io/library/busybox/1.29.json
  docker.io/library/busybox/1.30.json
  docker.io/library/busybox/1.31.json
  docker.io/library/busybox/1.32.json
  docker.io/library/busybox/1.33.j

In [ ]:
# Step 3 — run guts diff: tool container vs base OS images
from container_guts.main import ManifestGenerator

gen = ManifestGenerator(tech="singularity")
print(f"Running diff for: {cvmfs_path}")
print(f"Base DB: {tmpdir}\n")

result = gen.diff(cvmfs_path, database=tmpdir)
print(f"Result keys: {list(result.keys()) if result else 'empty'}")

Running diff for: /cvmfs/singularity.galaxyproject.org/all/star-fusion:star-fusion/1.0.0--pl5.22.0_0
Base DB: /tmp/shelley-guts-j_dabz64

Generating diff for /cvmfs/singularity.galaxyproject.org/all/star-fusion:star-fusion/1.0.0--pl5.22.0_0


There was an issue running singularity pull /cvmfs/singularity.galaxyproject.org/all/star-fusion:star-fusion/1.0.0--pl5.22.0_0


FATAL:   While pulling library image: error fetching image: request did not succeed: invalid container ref: cvmfs/singularity.galaxyproject.org/all/star-fusion (400 Bad Request)


SystemExit: 1

/home/ubuntu/shelley-bio/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Step 4 — inspect the raw diff structure
import json

first_key = next(iter(result))
diff_data = result[first_key].get("diff", {})

print(f"Top-level diff keys: {list(diff_data.keys())}\n")

# Show a sample of unique_paths before filtering
unique_paths_raw = diff_data.get("unique_paths", [])
print(f"unique_paths total: {len(unique_paths_raw)}")
print("First 20 entries:")
for p in sorted(unique_paths_raw)[:20]:
    print(" ", p)

In [ ]:
# Step 4b — break down unique_paths by directory prefix
from collections import Counter
import re

prefix_re = re.compile(r"^(/[^/]+/[^/]+)")  # first two path segments

prefixes = Counter()
for p in unique_paths_raw:
    m = prefix_re.match(p)
    if m:
        prefixes[m.group(1)] += 1

print("unique_paths by directory prefix:")
for prefix, count in prefixes.most_common():
    print(f"  {count:4d}  {prefix}")

In [ ]:
# Step 5 — filter to bin/sbin executables only
# unique_paths contains every file unique to the container (libs, data, configs…).
# shpc aliases should only be commands, so restrict to paths under a bin/ or sbin/ dir.
import re

bin_re = re.compile(r"/(s?bin)/")

all_paths = sorted(unique_paths_raw)
bin_paths = [p for p in all_paths if bin_re.search(p) and os.path.basename(p)]
basenames = [os.path.basename(p) for p in bin_paths]

print(f"Total unique_paths : {len(all_paths)}")
print(f"After bin filter   : {len(bin_paths)}")
print("\nExecutable names:")
for name in basenames:
    print(" ", name)

In [ ]:
# Step 6 — build shpc alias dicts (final output of extract_aliases)
aliases = [{"name": name, "command": name} for name in basenames]

print(f"shpc aliases ({len(aliases)}):")
for a in aliases:
    print(f"  {a}")

# Cleanup tmpdir from step 2
shutil.rmtree(tmpdir, ignore_errors=True)
print("\ntmpdir cleaned up.")